# 01 Exploratory Data Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math

In [ ]:
df = pd.read_csv("../data/raw/HR-Employee-Attrition.csv")
df.head()

In [ ]:
df.dtypes

In [ ]:
df.describe()

In [ ]:
df.isna().sum()

In [ ]:
(df.select_dtypes(include="number") < 0).any()

In [ ]:
df.select_dtypes(include="number").columns.size

In [ ]:
numeric_feats = df.select_dtypes(include="number").columns.values

In [ ]:
ordinal_feats = [
    "EnvironmentSatisfaction",
    "JobInvolvement",
    "JobSatisfaction",
    "PerformanceRating",
    "RelationshipSatisfaction",
    "WorkLifeBalance",
    "Education"
]

In [ ]:
numeric_feats = list(set(numeric_feats) - set(ordinal_feats))

In [ ]:
df_melt = df[numeric_feats + ["Attrition"]].melt(
    id_vars="Attrition", var_name="feature", value_name="value"
)

g = sns.catplot(
    data=df_melt,
    x="Attrition", y="value",
    col="feature", col_wrap=5,
    kind="violin",
    hue="Attrition",
    palette="muted",
    sharey=False,
    height=3.5, aspect=1
)

g.set_titles("{col_name}")
g.set_axis_labels("Attrition", "")
plt.tight_layout()
plt.show()

In [ ]:
df[ordinal_feats].describe()

In [ ]:
n_feats = len(ordinal_feats)
n_cols = 4
n_rows = math.ceil(n_feats / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows))
axes = axes.flatten()

for i, feat in enumerate(ordinal_feats):
    ct = pd.crosstab(df["Attrition"], df[feat], normalize="index")

    ct.plot(
        kind="bar",
        stacked=True,
        ax=axes[i],
        colormap="tab20",
        width=0.6,
        legend=False
    )

    axes[i].set_title(feat, fontsize=11, fontweight="bold")
    axes[i].set_xlabel("")
    axes[i].set_ylabel("Proporción")
    axes[i].set_ylim(0, 1)
    axes[i].tick_params(axis="x", rotation=0)

    axes[i].legend(
        fontsize=7,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.12),
        ncol=min(len(ct.columns), 3),
        frameon=False
    )

for j in range(n_feats, len(axes)):
    fig.delaxes(axes[j])

# fig.suptitle("Distribución de Attrition por categoría (100% Stacked Bar)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
df.select_dtypes(include="str").columns

In [ ]:
for col in df.select_dtypes(include="str").columns.values:
    print(df[col].value_counts())
    print(f"=================")

In [ ]:
n_rows, n_cols = 2, 4

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows))
axes = axes.flatten()

features = [
    "BusinessTravel",
    "Department",
    "EducationField",
    "Gender",
    "JobRole",
    "MaritalStatus",
    "Over18",
    "OverTime",
]

for i, feat in enumerate(features):
    ct = pd.crosstab(df["Attrition"], df[feat], normalize="index")

    ct.plot(
        kind="bar",
        stacked=True,
        ax=axes[i],
        colormap="tab20",
        width=0.7
    )

    axes[i].set_title(feat, fontsize=11, fontweight="bold")
    axes[i].set_xlabel("Attrition")
    axes[i].set_ylabel("Ratio")
    axes[i].set_ylim(0, 1)
    axes[i].legend(
        title=feat,
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        fontsize=8
    )
    axes[i].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
num_df = df.select_dtypes(include="number")

corr = num_df.corr()

plt.figure(figsize=(14, 12))
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    square=True,
    linewidths=0.5,
    annot_kws={"size": 7},
    cbar_kws={"shrink": 0.8}
)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

X = num_df.drop(columns=["Attrition", "EmployeeCount", "StandardHours"], errors="ignore")
X = add_constant(X)

vif_data = pd.DataFrame({
    "feature": X.columns,
    "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
})
vif_data = vif_data[vif_data["feature"] != "const"].sort_values("VIF", ascending=False)
vif_data